# ChestScan AI — MLflow Tracking & Model Evaluation

**Objectif :** suivre et comparer les expériences MLflow existantes sans relancer l'entraînement.

**À exécuter sur le noyau Python distant de la même VM Vast.ai** que le notebook d'entraînement.

- Ce notebook ne télécharge pas le dataset et n'appelle pas `model.fit()`.
- Il lit la base `mlflow.db`, les runs et les rapports existants.
- Les cellules qui écrivent de nouveaux résultats sont facultatives.
- Les chemins supposent que le projet est dans `/workspace/chestscan-ai-cnn`.


## 1 — Installer et importer les dépendances

In [28]:
%pip install "protobuf>=3.20.3,<6"

Note: you may need to restart the kernel to use updated packages.


In [29]:
%pip install mlflow pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [30]:
import tensorflow as tf
import mlflow
import google.protobuf

print("TensorFlow :", tf.__version__)
print("MLflow :", mlflow.__version__)
print("Protobuf :", google.protobuf.__version__)

print("GPU :", tf.config.list_physical_devices("GPU"))

# Vérifier que TensorFlow peut construire un modèle
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10,)),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

print("TensorFlow fonctionne.")

TensorFlow : 2.19.0
MLflow : 3.17.2
Protobuf : 5.29.6
GPU : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
TensorFlow fonctionne.


In [31]:
from pathlib import Path
import json
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

print("MLflow :", mlflow.__version__)

MLflow : 3.17.2


## 2 — Configurer le projet local

In [32]:
from pathlib import Path
import os
import mlflow

print("=== ENVIRONNEMENT ===")
print("Python :", os.sys.executable)
print("Dossier courant :", Path.cwd())
print("MLflow version :", mlflow.__version__)
print("Tracking URI actuel :", mlflow.get_tracking_uri())

print("\n=== DOSSIERS POSSIBLES DU PROJET ===")

for path in [
    Path("/workspace/chestscan-ai-cnn"),
    Path.cwd(),
    Path.home(),
    Path("/workspace"),
]:
    print(f"{path} | existe : {path.exists()}")

print("\n=== VARIABLES D'ENVIRONNEMENT MLFLOW ===")

for key, value in os.environ.items():
    if "MLFLOW" in key.upper():
        print(f"{key} = {value}")

print("\n=== FICHIERS MLFLOW À PROXIMITÉ ===")

search_roots = [
    Path.cwd(),
    Path.home(),
    Path("/workspace"),
]

found = set()

for root in search_roots:
    if root.exists():
        for pattern in ["mlflow.db", "mlruns", "mlflow.sqlite"]:
            try:
                for path in root.rglob(pattern):
                    if path not in found:
                        found.add(path)
                        print(path)
            except PermissionError:
                pass

if not found:
    print("Aucun fichier ou dossier MLflow trouvé dans ces emplacements.")

=== ENVIRONNEMENT ===
Python : /venv/main/bin/python
Dossier courant : /
MLflow version : 3.17.2
Tracking URI actuel : sqlite:////artifacts/mlflow/mlflow.db

=== DOSSIERS POSSIBLES DU PROJET ===
/workspace/chestscan-ai-cnn | existe : True
/ | existe : True
/root | existe : True
/workspace | existe : True

=== VARIABLES D'ENVIRONNEMENT MLFLOW ===
_MLFLOW_TELEMETRY_SESSION_ID = f0fda982d73c4c9ba4287bce08b32335
MLFLOW_TRACKING_URI = sqlite:////artifacts/mlflow/mlflow.db
MLFLOW_EXPERIMENT_ID = 1

=== FICHIERS MLFLOW À PROXIMITÉ ===
/mlflow.db
/root/.local/share/Trash/files/mlflow.db
/root/.local/share/Trash/files/artifacts.2/mlflow/mlflow.db
/mlruns


In [33]:
from pathlib import Path
import os
import mlflow
import pandas as pd
from mlflow.tracking import MlflowClient

# ============================================================
# 1. Reprendre la configuration MLflow existante
# ============================================================

PROJECT_DIR = Path("/workspace/chestscan-ai-cnn").resolve()
TRACKING_URI = os.environ.get(
    "MLFLOW_TRACKING_URI",
    mlflow.get_tracking_uri()
)
EXPERIMENT_ID = os.environ.get("MLFLOW_EXPERIMENT_ID", "1")

print("Projet :", PROJECT_DIR, "| Existe :", PROJECT_DIR.exists())
print("Tracking URI :", TRACKING_URI)
print("Experiment ID :", EXPERIMENT_ID)

# Ne pas remplacer le Tracking URI par une nouvelle base.
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient(tracking_uri=TRACKING_URI)

# ============================================================
# 2. Vérifier l'expérience existante
# ============================================================

experiment = client.get_experiment(EXPERIMENT_ID)

if experiment is None:
    raise ValueError(
        f"Expérience {EXPERIMENT_ID} introuvable dans {TRACKING_URI}"
    )

print("\nExpérience :", experiment.name)
print("ID :", experiment.experiment_id)
print("Statut :", experiment.lifecycle_stage)

# ============================================================
# 3. Consulter les Runs existants et leurs métriques
# ============================================================

runs = client.search_runs(
    experiment_ids=[EXPERIMENT_ID],
    order_by=["attributes.start_time DESC"],
    max_results=100
)

print(f"\nNombre de Runs récupérés : {len(runs)}")

rows = []

for run in runs:
    rows.append({
        "run_name": run.info.run_name,
        "run_id": run.info.run_id,
        "status": run.info.status,
        "start_time": run.info.start_time,
        **run.data.params,
        **{
            f"metric_{key}": value
            for key, value in run.data.metrics.items()
        }
    })

df_runs = pd.DataFrame(rows)

if not df_runs.empty:
    display(df_runs)
else:
    print("Aucun Run trouvé dans cette expérience.")

Projet : /workspace/chestscan-ai-cnn | Existe : True
Tracking URI : sqlite:////artifacts/mlflow/mlflow.db
Experiment ID : 1

Expérience : ChestScan_AI
ID : 1
Statut : active

Nombre de Runs récupérés : 5


,run_name,run_id,status,start_time,model,task,model_file,model_size_mb
0,MobileNetV2_FineTuned,c9577e4377434232aecb8a76acbdcec7,FAILED,1793032350684,MobileNetV2,binary_image_classification,NaN,NaN
1,existing_reports,1c13eb410a704638950f0357ce1c82cf,FINISHED,1793031054384,NaN,NaN,NaN,NaN
2,Model_tracking,af194530a4de45c797d4168a92d9f9a6,FINISHED,1793031039631,NaN,NaN,model.keras,22.607
3,MobileNetV2_FineTuned_tracking,737d40da4abd4386a9d02d74b8d9db9b,FINISHED,1793031039616,NaN,NaN,NaN,NaN
4,MobileNetV2_tracking,ae062ad9562a4cbfad3537a63d69b78b,FINISHED,1793031039582,NaN,NaN,NaN,NaN


## 3 — Créer ou récupérer l'expérience

In [34]:
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)

if experiment is None:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
else:
    experiment_id = experiment.experiment_id

mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient()

print("Expérience :", EXPERIMENT_NAME)
print("Experiment ID :", experiment_id)
print("Tracking URI :", mlflow.get_tracking_uri())

Expérience : ChestScan_AI
Experiment ID : 1
Tracking URI : sqlite:////artifacts/mlflow/mlflow.db


## 4 — Inventorier les modèles et rapports existants

In [35]:
from pathlib import Path

# Vérification des modèles sauvegardés
model_files = [
    MODEL_DIR / "mobilenetv2_best.keras",
    MODEL_DIR / "mobilenetv2_finetuned_best.keras",
    MODEL_DIR / "model.keras",
    MODEL_DIR / "CNN_v1.keras",
    MODEL_DIR / "MobileNetV2_FineTuned.keras",
]

print("=== Modèles sauvegardés ===")

existing_models = []

for path in model_files:
    if path.exists():
        size_mb = path.stat().st_size / (1024 ** 2)
        print(f"✅ {path.name} ({size_mb:.2f} MB)")
        existing_models.append(path)
    else:
        print(f"❌ Absent : {path.name}")

# Vérification des rapports
print("\n=== Rapports existants ===")

if REPORT_DIR.exists():
    report_files = sorted(
        path for path in REPORT_DIR.iterdir()
        if path.is_file()
    )

    if report_files:
        for path in report_files:
            size_kb = path.stat().st_size / 1024
            print(f"- {path.name} ({size_kb:.2f} KB)")
    else:
        print("Aucun rapport trouvé.")
else:
    print(f"Répertoire absent : {REPORT_DIR}")

# Résumé
print("\n=== Résumé ===")
print(f"Modèles disponibles : {len(existing_models)}")
print(f"Répertoire des modèles : {MODEL_DIR}")
print(f"Répertoire des rapports : {REPORT_DIR}")

=== Modèles sauvegardés ===
❌ Absent : mobilenetv2_best.keras
❌ Absent : mobilenetv2_finetuned_best.keras
❌ Absent : model.keras
❌ Absent : CNN_v1.keras
❌ Absent : MobileNetV2_FineTuned.keras

=== Rapports existants ===
Aucun rapport trouvé.

=== Résumé ===
Modèles disponibles : 0
Répertoire des modèles : /artifacts/models
Répertoire des rapports : /artifacts/reports


## 5 — Définir les métriques disponibles
Nous allons réutiliser les rapports déjà produits par ton notebook d'entraînement, plutôt que recalculer les prédictions.

## 3. Afficher les runs existants

In [36]:
def load_metrics_from_csv(path: Path) -> dict:
    if not path.exists():
        return {}

    df = pd.read_csv(path)

    if df.empty:
        return {}

    row = df.iloc[0]
    metrics = {}

    for column, value in row.items():
        try:
            number = float(value)
            if pd.notna(number):
                # MLflow attend des métriques numériques finies.
                import math
                if math.isfinite(number):
                    metrics[column] = number
        except (ValueError, TypeError):
            continue

    return metrics

## 6 — Enregistrer les résultats des modèles dans MLflow
Les noms de fichiers de rapports doivent correspondre à ceux réellement présents dans artifacts/reports/. Adapte les chemins ci-dessous si nécessaire.

In [38]:
from pathlib import Path
import os
import mlflow

db_path = Path("/artifacts/mlflow/mlflow.db")
db_dir = db_path.parent

print("Tracking URI :", mlflow.get_tracking_uri())
print("Base existe :", db_path.exists())
print("Dossier existe :", db_dir.exists())

if db_path.exists():
    print("Base accessible en écriture :", os.access(db_path, os.W_OK))

if db_dir.exists():
    print("Dossier accessible en écriture :", os.access(db_dir, os.W_OK))

try:
    test_file = db_dir / ".permission_test"
    test_file.write_text("test")
    test_file.unlink()
    print("Écriture dans le dossier : OK")
except OSError as e:
    print("Écriture dans le dossier impossible :", repr(e))

Tracking URI : sqlite:////artifacts/mlflow/mlflow.db
Base existe : False
Dossier existe : True
Dossier accessible en écriture : True
Écriture dans le dossier : OK


In [40]:
from pathlib import Path
import os

candidates = [
    Path("/mlflow.db"),
    Path("/artifacts/mlflow/mlflow.db"),
    Path("/root/.local/share/Trash/files/mlflow.db"),
    Path("/root/.local/share/Trash/files/artifacts.2/mlflow/mlflow.db"),
]

for path in candidates:
    print(f"\nBase : {path}")
    print("Existe :", path.is_file())

    if path.is_file():
        print("Taille (Mo) :", round(path.stat().st_size / 1024**2, 3))
        print("Lecture autorisée :", os.access(path, os.R_OK))
        print("Écriture autorisée :", os.access(path, os.W_OK))

        for suffix in ("-wal", "-shm"):
            sidecar = Path(str(path) + suffix)
            if sidecar.exists():
                print(f"Fichier associé présent : {sidecar}")


Base : /mlflow.db
Existe : True
Taille (Mo) : 0.91
Lecture autorisée : True
Écriture autorisée : True

Base : /artifacts/mlflow/mlflow.db
Existe : False

Base : /root/.local/share/Trash/files/mlflow.db
Existe : True
Taille (Mo) : 0.91
Lecture autorisée : True
Écriture autorisée : True

Base : /root/.local/share/Trash/files/artifacts.2/mlflow/mlflow.db
Existe : True
Taille (Mo) : 0.91
Lecture autorisée : True
Écriture autorisée : True


In [41]:
import sqlite3
from pathlib import Path

db_path = Path("/mlflow.db")

if not db_path.is_file():
    raise FileNotFoundError(db_path)

connection = sqlite3.connect(
    f"{db_path.resolve().as_uri()}?mode=ro",
    uri=True
)

try:
    tables = {
        row[0]
        for row in connection.execute(
            "SELECT name FROM sqlite_master WHERE type='table'"
        )
    }

    print("Tables MLflow détectées :", sorted(tables))

    if "experiments" in tables:
        print("\n=== Expériences ===")
        for row in connection.execute(
            "SELECT experiment_id, name, lifecycle_stage FROM experiments"
        ):
            print(row)

    if "runs" in tables:
        print("\n=== Nombre de runs ===")
        print(connection.execute("SELECT COUNT(*) FROM runs").fetchone()[0])

        print("\n=== Runs récents ===")
        for row in connection.execute("""
            SELECT run_uuid, experiment_id, status, start_time, artifact_uri
            FROM runs
            ORDER BY start_time DESC
            LIMIT 10
        """):
            print(row)
finally:
    connection.close()

Tables MLflow détectées : ['alembic_version', 'assessments', 'budget_policies', 'datasets', 'endpoint_bindings', 'endpoint_model_mappings', 'endpoint_tags', 'endpoints', 'entity_associations', 'evaluation_dataset_records', 'evaluation_dataset_tags', 'evaluation_datasets', 'experiment_tags', 'experiments', 'guardrail_configs', 'guardrails', 'input_tags', 'inputs', 'issues', 'job_locks', 'jobs', 'label_schemas', 'latest_metrics', 'logged_model_metrics', 'logged_model_params', 'logged_model_tags', 'logged_models', 'mcp_access_endpoints', 'mcp_server_aliases', 'mcp_server_tags', 'mcp_server_version_tags', 'mcp_server_versions', 'mcp_servers', 'metrics', 'model_definitions', 'model_version_tags', 'model_versions', 'online_scoring_configs', 'params', 'registered_model_aliases', 'registered_model_tags', 'registered_models', 'review_queue_items', 'review_queue_label_schemas', 'review_queue_users', 'review_queues', 'runs', 'scheduler_leases', 'scorer_versions', 'scorers', 'secrets', 'span_metri

In [42]:
import os
import mlflow
from mlflow.tracking import MlflowClient

# Utiliser la base existante, sans créer une nouvelle expérience
mlflow.set_tracking_uri("sqlite:////mlflow.db")
client = MlflowClient()

experiment = client.get_experiment("1")

print("Tracking URI :", mlflow.get_tracking_uri())
print("Expérience :", experiment.name)
print("Experiment ID :", experiment.experiment_id)

runs = client.search_runs(
    experiment_ids=["1"],
    max_results=100
)

print("\nNombre de runs :", len(runs))

for run in runs:
    artifact_uri = run.info.artifact_uri
    print("\nRun ID :", run.info.run_id)
    print("Statut :", run.info.status)
    print("Artifact URI :", artifact_uri)

    if artifact_uri.startswith("file://"):
        artifact_path = artifact_uri.removeprefix("file://")
        print("Dossier artefacts accessible :", os.path.isdir(artifact_path))
    else:
        print("URI d'artefacts :", artifact_uri)

Tracking URI : sqlite:////mlflow.db
Expérience : ChestScan-AI-Model-Comparison
Experiment ID : 1

Nombre de runs : 2

Run ID : c836798334004d50ab7995f352b3410c
Statut : FINISHED
Artifact URI : /mlruns/1/c836798334004d50ab7995f352b3410c/artifacts
URI d'artefacts : /mlruns/1/c836798334004d50ab7995f352b3410c/artifacts

Run ID : 2b01268b686b491da1bd5907317092e5
Statut : FINISHED
Artifact URI : /mlruns/1/2b01268b686b491da1bd5907317092e5/artifacts
URI d'artefacts : /mlruns/1/2b01268b686b491da1bd5907317092e5/artifacts


In [43]:
from pathlib import Path
from mlflow.tracking import MlflowClient

client = MlflowClient()

print("=== Artefacts des runs existants ===")

for run in client.search_runs(
    experiment_ids=["1"],
    max_results=100
):
    print(f"\nRun : {run.info.run_id}")

    try:
        artifacts = client.list_artifacts(run.info.run_id)
        for artifact in artifacts:
            print(" -", artifact.path)
        if not artifacts:
            print(" Aucun artefact à la racine du run.")
    except Exception as e:
        print(" Erreur d'accès aux artefacts :", repr(e))

print("\n=== Fichiers des modèles ===")

for name, config in model_configs.items():
    path = Path(config["model_file"])
    print(
        f"{name} : existe={path.is_file()}, "
        f"chemin={path}"
    )

=== Artefacts des runs existants ===

Run : c836798334004d50ab7995f352b3410c
 - MobileNetV2_FineTuned_confusion_matrix.json
 - MobileNetV2_FineTuned_test_report.json
 - model_file

Run : 2b01268b686b491da1bd5907317092e5
 - CNN_v1_confusion_matrix.json
 - CNN_v1_test_report.json
 - model_file

=== Fichiers des modèles ===
MobileNetV2 : existe=False, chemin=/artifacts/models/mobilenetv2_best.keras
MobileNetV2_FineTuned : existe=False, chemin=/artifacts/models/mobilenetv2_finetuned_best.keras
Model : existe=False, chemin=/artifacts/models/model.keras


In [44]:
import mlflow

mlflow.set_tracking_uri("sqlite:////mlflow.db")
mlflow.set_experiment("ChestScan-AI-Model-Comparison")

<Experiment: artifact_location='/mlruns/1', creation_time=1793035196536, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1793035196536, lifecycle_stage='active', name='ChestScan-AI-Model-Comparison', tags={}, trace_location=None, workspace='default'>

## 7 — Enregistrer les rapports disponibles
Cette cellule ajoute les rapports de ton projet aux artefacts MLflow.

In [48]:
from pathlib import Path

print("REPORT_DIR =", REPORT_DIR)
print("Dossier existe :", REPORT_DIR.exists())

if REPORT_DIR.exists():
    print("\nContenu du dossier :")

    for p in REPORT_DIR.iterdir():
        print(
            "Dossier" if p.is_dir() else "Fichier",
            ":", p.name
        )
else:
    print("Le chemin configuré est incorrect ou inaccessible.")

REPORT_DIR = /artifacts/reports
Dossier existe : True

Contenu du dossier :


In [49]:
from pathlib import Path

PROJECT_DIR = Path("/workspace/chestscan-ai-cnn")

extensions = {".csv", ".json", ".txt"}

report_candidates = [
    p for p in PROJECT_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
]

print("Nombre de fichiers trouvés :", len(report_candidates))

for p in report_candidates:
    print(p)

Nombre de fichiers trouvés : 8
/workspace/chestscan-ai-cnn/artifacts/reports/dataset_audit.csv
/workspace/chestscan-ai-cnn/artifacts/reports/test_model_comparison.csv
/workspace/chestscan-ai-cnn/artifacts/reports/mobilenet_threshold_comparison.csv
/workspace/chestscan-ai-cnn/artifacts/reports/validation_threshold_results.csv
/workspace/chestscan-ai-cnn/artifacts/reports/CNN_v1_test_report.json
/workspace/chestscan-ai-cnn/artifacts/reports/CNN_v1_confusion_matrix.json
/workspace/chestscan-ai-cnn/artifacts/reports/MobileNetV2_FineTuned_test_report.json
/workspace/chestscan-ai-cnn/artifacts/reports/MobileNetV2_FineTuned_confusion_matrix.json


In [50]:
import mlflow

mlflow.set_tracking_uri("sqlite:////mlflow.db")
mlflow.set_experiment("ChestScan-AI-Model-Comparison")

with mlflow.start_run(run_name="existing_reports") as run:
    mlflow.set_tag("project", "ChestScan AI")
    mlflow.set_tag("run_purpose", "report_archiving")

    for report_path in report_candidates:
        mlflow.log_artifact(
            str(report_path),
            artifact_path="reports"
        )

    print("Rapports archivés :", len(report_candidates))
    print("Run ID :", run.info.run_id)

Rapports archivés : 8
Run ID : 223703e3b66c4c61acf88213c5186234


## 8 — Comparer les runs

In [46]:
runs = mlflow.search_runs(
    experiment_ids=[experiment_id],
    order_by=["start_time DESC"]
)

display_columns = [
    column for column in [
        "run_id",
        "tags.mlflow.runName",
        "metrics.accuracy",
        "metrics.precision",
        "metrics.recall",
        "metrics.f1",
        "metrics.auc",
        "start_time"
    ]
    if column in runs.columns
]

display(runs[display_columns])

,run_id,tags.mlflow.runName,metrics.accuracy,metrics.precision,metrics.recall,start_time
0,895b07993e34478e8436ae7f280d6a71,existing_reports,NaN,NaN,NaN,2026-10-27 08:55:57.733000+00:00
1,c836798334004d50ab7995f352b3410c,MobileNetV2_FineTuned,0.817308,0.774900,0.997436,2026-10-26 17:20:00.065000+00:00
2,2b01268b686b491da1bd5907317092e5,CNN_v1,0.695513,0.679211,0.971795,2026-10-26 17:19:57.487000+00:00


In [51]:
import mlflow
from mlflow.tracking import MlflowClient

mlflow.set_tracking_uri("sqlite:////mlflow.db")
client = MlflowClient()

experiments = client.search_experiments()

print("Expériences MLflow disponibles :")

for exp in experiments:
    print(
        f"Nom : {exp.name} | "
        f"ID : {exp.experiment_id} | "
        f"Statut : {exp.lifecycle_stage}"
    )

Expériences MLflow disponibles :
Nom : ChestScan-AI-Model-Comparison | ID : 1 | Statut : active
Nom : Default | ID : 0 | Statut : active


In [52]:
experiment = client.get_experiment_by_name(
    "ChestScan-AI-Model-Comparison"
)

if experiment is None:
    raise ValueError(
        "Expérience introuvable. Vérifiez le tracking URI."
    )

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=100
)

metric_columns = [
    column
    for column in runs.columns
    if column.startswith("metrics.")
]

print("Nombre de runs :", len(runs))
print("Métriques enregistrées :")

for column in metric_columns:
    print("-", column)

display(
    runs[["run_id", "tags.mlflow.runName"] + metric_columns]
)

Nombre de runs : 4
Métriques enregistrées :
- metrics.false_negatives
- metrics.recall
- metrics.true_positives
- metrics.false_positives
- metrics.roc_auc
- metrics.specificity
- metrics.accuracy
- metrics.f1_score
- metrics.precision
- metrics.true_negatives


,run_id,tags.mlflow.runName,metrics.false_negatives,metrics.recall,metrics.true_positives,metrics.false_positives,metrics.roc_auc,metrics.specificity,metrics.accuracy,metrics.f1_score,metrics.precision,metrics.true_negatives
0,223703e3b66c4c61acf88213c5186234,existing_reports,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,895b07993e34478e8436ae7f280d6a71,existing_reports,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,c836798334004d50ab7995f352b3410c,MobileNetV2_FineTuned,1.0,0.997436,389.0,113.0,0.967467,0.517094,0.817308,0.872197,0.774900,121.0
3,2b01268b686b491da1bd5907317092e5,CNN_v1,11.0,0.971795,379.0,179.0,0.898773,0.235043,0.695513,0.799578,0.679211,55.0


In [54]:
from pathlib import Path

PROJECT_DIR = Path("/workspace/chestscan-ai-cnn")

for folder_name in ["artifacts", "reports"]:
    folder = PROJECT_DIR / folder_name

    print(f"\n--- {folder} ---")

    if folder.exists():
        for path in folder.rglob("*"):
            if path.is_file():
                print(path.relative_to(PROJECT_DIR))
    else:
        print("Dossier introuvable")


--- /workspace/chestscan-ai-cnn/artifacts ---
artifacts/models/CNN_v1.keras
artifacts/models/MobileNetV2_FineTuned.keras
artifacts/models/model.keras
artifacts/reports/dataset_audit.csv
artifacts/reports/test_model_comparison.csv
artifacts/reports/mobilenet_threshold_comparison.csv
artifacts/reports/validation_threshold_results.csv
artifacts/reports/CNN_v1_test_report.json
artifacts/reports/CNN_v1_confusion_matrix.json
artifacts/reports/MobileNetV2_FineTuned_test_report.json
artifacts/reports/MobileNetV2_FineTuned_confusion_matrix.json

--- /workspace/chestscan-ai-cnn/reports ---
Dossier introuvable


In [55]:

from pathlib import Path
import json
import pandas as pd

PROJECT_DIR = Path("/workspace/chestscan-ai-cnn")
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"

json_files = [
    "CNN_v1_test_report.json",
    "MobileNetV2_FineTuned_test_report.json",
    "CNN_v1_confusion_matrix.json",
    "MobileNetV2_FineTuned_confusion_matrix.json",
]

for filename in json_files:
    path = REPORT_DIR / filename
    print(f"\n{'=' * 60}")
    print(filename)

    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        print(json.dumps(data, indent=2, ensure_ascii=False))
    else:
        print("Fichier introuvable")

csv_files = [
    "test_model_comparison.csv",
    "mobilenet_threshold_comparison.csv",
    "validation_threshold_results.csv",
]

for filename in csv_files:
    path = REPORT_DIR / filename
    print(f"\n{'=' * 60}")
    print(filename)

    if path.exists():
        df = pd.read_csv(path)
        print("Colonnes :", df.columns.tolist())
        display(df.head(10))
    else:
        print("Fichier introuvable")



CNN_v1_test_report.json
{
  "NORMAL": {
    "precision": 0.8333333333333334,
    "recall": 0.23504273504273504,
    "f1-score": 0.36666666666666664,
    "support": 234.0
  },
  "PNEUMONIA": {
    "precision": 0.6792114695340502,
    "recall": 0.9717948717948718,
    "f1-score": 0.79957805907173,
    "support": 390.0
  },
  "accuracy": 0.6955128205128205,
  "macro avg": {
    "precision": 0.7562724014336918,
    "recall": 0.6034188034188034,
    "f1-score": 0.5831223628691983,
    "support": 624.0
  },
  "weighted avg": {
    "precision": 0.7370071684587813,
    "recall": 0.6955128205128205,
    "f1-score": 0.6372362869198313,
    "support": 624.0
  }
}

MobileNetV2_FineTuned_test_report.json
{
  "NORMAL": {
    "precision": 0.9918032786885246,
    "recall": 0.5170940170940171,
    "f1-score": 0.6797752808988764,
    "support": 234.0
  },
  "PNEUMONIA": {
    "precision": 0.7749003984063745,
    "recall": 0.9974358974358974,
    "f1-score": 0.8721973094170403,
    "support": 390.0
  },

,Model,Threshold,Accuracy,Precision,Recall,F1-score,Specificity,ROC-AUC,TN,FP,FN,TP
0,MobileNetV2_FineTuned,0.6,0.817308,0.774900,0.997436,0.872197,0.517094,0.967691,121,113,1,389
1,CNN_v1,0.6,0.695513,0.679211,0.971795,0.799578,0.235043,0.898795,55,179,11,379



mobilenet_threshold_comparison.csv
Colonnes : ['Threshold', 'Accuracy', 'Precision', 'Recall_PNEUMONIA', 'Recall_NORMAL', 'Specificity', 'F1-score', 'TN', 'FP', 'FN', 'TP']


,Threshold,Accuracy,Precision,Recall_PNEUMONIA,Recall_NORMAL,Specificity,F1-score,TN,FP,FN,TP
0,0.3,0.980898,0.979721,0.994852,0.940741,0.940741,0.987229,254,16,4,773
1,0.4,0.979943,0.982143,0.990991,0.948148,0.948148,0.986547,256,14,7,770
2,0.5,0.979943,0.985861,0.987130,0.959259,0.959259,0.986495,259,11,10,767
3,0.6,0.982808,0.989677,0.987130,0.970370,0.970370,0.988402,262,8,10,767
4,0.7,0.981853,0.992208,0.983269,0.977778,0.977778,0.987718,264,6,13,764



validation_threshold_results.csv
Colonnes : ['Model', 'Threshold', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'Specificity', 'ROC-AUC', 'TN', 'FP', 'FN', 'TP']


,Model,Threshold,Accuracy,Precision,Recall,F1-score,Specificity,ROC-AUC,TN,FP,FN,TP
0,CNN_v1,0.10,0.743075,0.743295,0.998713,0.852279,0.007407,0.973998,2,268,1,776
1,CNN_v1,0.15,0.749761,0.748792,0.997426,0.855408,0.037037,0.973998,10,260,2,775
2,CNN_v1,0.20,0.767908,0.762795,0.997426,0.864473,0.107407,0.973998,29,241,2,775
3,CNN_v1,0.25,0.800382,0.789206,0.997426,0.881182,0.233333,0.973998,63,207,2,775
4,CNN_v1,0.30,0.829035,0.815401,0.994852,0.896232,0.351852,0.973998,95,175,4,773
5,CNN_v1,0.35,0.862464,0.847420,0.993565,0.914692,0.485185,0.973998,131,139,5,772
6,CNN_v1,0.40,0.878701,0.867647,0.987130,0.923540,0.566667,0.973998,153,117,10,767
7,CNN_v1,0.45,0.890162,0.884884,0.979408,0.929750,0.633333,0.973998,171,99,16,761
8,CNN_v1,0.50,0.904489,0.906363,0.971686,0.937888,0.711111,0.973998,192,78,22,755
9,CNN_v1,0.55,0.914995,0.926799,0.961390,0.943778,0.781481,0.973998,211,59,30,747


In [56]:

from pathlib import Path
import pandas as pd
import mlflow
from mlflow.tracking import MlflowClient

PROJECT_DIR = Path("/workspace/chestscan-ai-cnn")
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"
COMPARISON_FILE = REPORT_DIR / "test_model_comparison.csv"

# Utiliser la base MLflow existante
mlflow.set_tracking_uri("sqlite:////mlflow.db")
mlflow.set_experiment("ChestScan-AI-Model-Comparison")

client = MlflowClient()
experiment = client.get_experiment_by_name(
    "ChestScan-AI-Model-Comparison"
)

if not COMPARISON_FILE.is_file():
    raise FileNotFoundError(COMPARISON_FILE)

df = pd.read_csv(COMPARISON_FILE)

metric_mapping = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1-score": "f1_score",
    "Specificity": "specificity",
    "ROC-AUC": "roc_auc",
    "TN": "true_negatives",
    "FP": "false_positives",
    "FN": "false_negatives",
    "TP": "true_positives",
}

for _, row in df.iterrows():
    model_name = str(row["Model"])
    threshold = float(row["Threshold"])

    with mlflow.start_run(
        run_name=f"{model_name}_test_metrics_import"
    ) as run:
        mlflow.set_tags({
            "project": "ChestScan AI",
            "model_name": model_name,
            "evaluation_split": "test",
            "metric_source": "test_model_comparison.csv",
            "tracking_mode": "metrics_import",
        })

        mlflow.log_param("threshold", threshold)

        for source_col, metric_name in metric_mapping.items():
            value = row.get(source_col)

            if pd.notna(value):
                mlflow.log_metric(
                    metric_name,
                    float(value)
                )

        print(
            f"Modèle : {model_name} | "
            f"Run ID : {run.info.run_id}"
        )

print("Importation des métriques terminée.")


Modèle : MobileNetV2_FineTuned | Run ID : f0022cd803334b859f2f1e807378ed4b
Modèle : CNN_v1 | Run ID : c573760f6e514596a641b32aebb8a3f9
Importation des métriques terminée.


In [57]:

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=100
)

columns = [
    "run_id",
    "tags.mlflow.runName",
    "tags.model_name",
    "tags.evaluation_split",
    "metrics.accuracy",
    "metrics.precision",
    "metrics.recall",
    "metrics.f1_score",
    "metrics.roc_auc",
]

display(runs[[c for c in columns if c in runs.columns]])


,run_id,tags.mlflow.runName,tags.model_name,tags.evaluation_split,metrics.accuracy,metrics.precision,metrics.recall,metrics.f1_score,metrics.roc_auc
0,c573760f6e514596a641b32aebb8a3f9,CNN_v1_test_metrics_import,CNN_v1,test,0.695513,0.679211,0.971795,0.799578,0.898795
1,f0022cd803334b859f2f1e807378ed4b,MobileNetV2_FineTuned_test_metrics_import,MobileNetV2_FineTuned,test,0.817308,0.774900,0.997436,0.872197,0.967691
2,223703e3b66c4c61acf88213c5186234,existing_reports,None,None,NaN,NaN,NaN,NaN,NaN
3,895b07993e34478e8436ae7f280d6a71,existing_reports,None,None,NaN,NaN,NaN,NaN,NaN
4,c836798334004d50ab7995f352b3410c,MobileNetV2_FineTuned,None,None,0.817308,0.774900,0.997436,0.872197,0.967467
5,2b01268b686b491da1bd5907317092e5,CNN_v1,None,None,0.695513,0.679211,0.971795,0.799578,0.898773


In [58]:

from pathlib import Path

PROJECT_DIR = Path("/workspace/chestscan-ai-cnn")

for folder in [
    PROJECT_DIR / "artifacts" / "models",
    PROJECT_DIR / "artifacts" / "reports",
]:
    print(f"\n--- {folder} ---")
    for path in sorted(folder.iterdir()):
        if path.is_file():
            print(path.name)



--- /workspace/chestscan-ai-cnn/artifacts/models ---
CNN_v1.keras
MobileNetV2_FineTuned.keras
model.keras

--- /workspace/chestscan-ai-cnn/artifacts/reports ---
CNN_v1_confusion_matrix.json
CNN_v1_test_report.json
MobileNetV2_FineTuned_confusion_matrix.json
MobileNetV2_FineTuned_test_report.json
dataset_audit.csv
mobilenet_threshold_comparison.csv
test_model_comparison.csv
validation_threshold_results.csv


In [59]:

import mlflow
from pathlib import Path

mlflow.set_tracking_uri("sqlite:////mlflow.db")
mlflow.set_experiment("ChestScan-AI-Model-Comparison")

model_path = Path(
    "/workspace/chestscan-ai-cnn/artifacts/models/model.keras"
)

if not model_path.is_file():
    raise FileNotFoundError(model_path)

with mlflow.start_run(run_name="final_model_selection") as run:
    mlflow.set_tags({
        "project": "ChestScan AI",
        "run_purpose": "final_model_selection",
        "model_role": "production_candidate",
        "model_file": "model.keras",
        "selection_status": "selected",
    })

    mlflow.log_param("model_file", model_path.name)
    mlflow.log_param(
        "model_size_mb",
        round(model_path.stat().st_size / (1024**2), 2)
    )

    mlflow.log_artifact(
        str(model_path),
        artifact_path="final_model"
    )

    print("Modèle final documenté dans MLflow.")
    print("Run ID :", run.info.run_id)


Modèle final documenté dans MLflow.
Run ID : 1e0a749736a54bcc9ff11be7c7e0fcad


In [60]:

import hashlib
from pathlib import Path
from mlflow.tracking import MlflowClient

client = MlflowClient()

run_id = "1e0a749736a54bcc9ff11be7c7e0fcad"

print("Run status :", client.get_run(run_id).info.status)

artifacts = client.list_artifacts(run_id, "final_model")

for artifact in artifacts:
    print(
        "Artifact :", artifact.path,
        "| Taille :", artifact.file_size, "bytes"
    )

def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

model_dir = Path("/workspace/chestscan-ai-cnn/artifacts/models")

for filename in ["model.keras", "MobileNetV2_FineTuned.keras"]:
    path = model_dir / filename
    print(f"\n{filename}")
    print("Existe :", path.is_file())
    if path.is_file():
        print("SHA-256 :", sha256_file(path))


Run status : FINISHED
Artifact : final_model/model.keras | Taille : 23704816 bytes

model.keras
Existe : True
SHA-256 : edb3e0497643abde7c37f169d6b9bf98d1c2d382b90557f6a7855679ed9a7c6f

MobileNetV2_FineTuned.keras
Existe : True
SHA-256 : 5eeab04ae67fcb47d6e4abb4e22f72efd39a45d0d2b36f3261718e04c612b2e2


In [65]:

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from pathlib import Path

model_dir = Path(
    "/workspace/chestscan-ai-cnn/artifacts/models"
)

model_paths = {
    "model.keras": model_dir / "model.keras",
    "MobileNetV2_FineTuned.keras":
        model_dir / "MobileNetV2_FineTuned.keras",
}

models = {}

for name, path in model_paths.items():
    print(f"\n--- Chargement : {name} ---")

    model = tf.keras.models.load_model(
        path,
        custom_objects={
            "preprocess_input": preprocess_input
        },
        compile=False,
        safe_mode=True
    )

    models[name] = model

    print("Architecture :", model.name)
    print("Input shape :", model.input_shape)
    print("Output shape :", model.output_shape)
    print("Paramètres :", model.count_params())



--- Chargement : model.keras ---
Architecture : chestscan_mobilenetv2_v1
Input shape : (None, 180, 180, 3)
Output shape : (None, 1)
Paramètres : 2422081

--- Chargement : MobileNetV2_FineTuned.keras ---
Architecture : chestscan_mobilenetv2_v1
Input shape : (None, 180, 180, 3)
Output shape : (None, 1)
Paramètres : 2422081


In [66]:

import mlflow
from mlflow.tracking import MlflowClient

# Utiliser la base MLflow existante
mlflow.set_tracking_uri("sqlite:////mlflow.db")

# Sélectionner l'expérience existante
experiment_name = "ChestScan-AI-Model-Comparison"
experiment = mlflow.get_experiment_by_name(experiment_name)

if experiment is None:
    raise ValueError(
        f"Expérience introuvable : {experiment_name}"
    )

mlflow.set_experiment(experiment_name)

client = MlflowClient()

print("MLflow Tracking URI :", mlflow.get_tracking_uri())
print("Nom de l'expérience :", experiment.name)
print("Experiment ID :", experiment.experiment_id)
print("Nombre de runs :", len(
    client.search_runs(
        experiment_ids=[experiment.experiment_id],
        max_results=100
    )
))


MLflow Tracking URI : sqlite:////mlflow.db
Nom de l'expérience : ChestScan-AI-Model-Comparison
Experiment ID : 1
Nombre de runs : 7


In [67]:

runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=100
)

for run in runs:
    print("-" * 60)
    print("Run ID :", run.info.run_id)
    print("Run name :", run.data.tags.get("mlflow.runName", "Sans nom"))
    print("Status :", run.info.status)
    print("Metrics :", run.data.metrics)


------------------------------------------------------------
Run ID : 1e0a749736a54bcc9ff11be7c7e0fcad
Run name : final_model_selection
Status : FINISHED
Metrics : {}
------------------------------------------------------------
Run ID : c573760f6e514596a641b32aebb8a3f9
Run name : CNN_v1_test_metrics_import
Status : FINISHED
Metrics : {'accuracy': 0.6955128205128205, 'precision': 0.6792114695340502, 'recall': 0.9717948717948718, 'f1_score': 0.79957805907173, 'specificity': 0.235042735042735, 'roc_auc': 0.8987946526408064, 'true_negatives': 55.0, 'false_positives': 179.0, 'false_negatives': 11.0, 'true_positives': 379.0}
------------------------------------------------------------
Run ID : f0022cd803334b859f2f1e807378ed4b
Run name : MobileNetV2_FineTuned_test_metrics_import
Status : FINISHED
Metrics : {'accuracy': 0.8173076923076923, 'precision': 0.7749003984063745, 'recall': 0.9974358974358974, 'f1_score': 0.8721973094170403, 'specificity': 0.5170940170940171, 'roc_auc': 0.9676912119219

In [62]:

import tensorflow as tf
from pathlib import Path

model_dir = Path(
    "/workspace/chestscan-ai-cnn/artifacts/models"
)

model_paths = {
    "model.keras": model_dir / "model.keras",
    "MobileNetV2_FineTuned.keras":
        model_dir / "MobileNetV2_FineTuned.keras",
}

models = {}

for name, path in model_paths.items():
    print(f"\n--- {name} ---")
    model = tf.keras.models.load_model(
        path,
        compile=False,
        safe_mode=True
    )
    models[name] = model

    print("Architecture :", model.name)
    print("Input shape  :", model.input_shape)
    print("Output shape :", model.output_shape)
    print("Parameters   :", model.count_params())

a = models["model.keras"]
b = models["MobileNetV2_FineTuned.keras"]

print("\n--- Comparaison ---")
print("Même architecture JSON :", a.to_json() == b.to_json())
print("Même nombre de poids   :", len(a.weights) == len(b.weights))

if len(a.weights) == len(b.weights):
    print(
        "Même forme des poids :",
        all(x.shape == y.shape for x, y in zip(a.weights, b.weights))
    )



--- model.keras ---


TypeError: <class 'keras.src.models.functional.Functional'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.src.models.functional', 'class_name': 'Functional', 'config': {}, 'registered_name': 'Functional', 'build_config': {'input_shape': None}, 'compile_config': None}.

Exception encountered: <class 'keras.src.layers.core.lambda_layer.Lambda'> could not be deserialized properly. Please ensure that components that are Python object instances (layers, models, etc.) returned by `get_config()` are explicitly deserialized in the model's `from_config()` method.

config={'module': 'keras.layers', 'class_name': 'Lambda', 'config': {'name': 'mobilenet_preprocessing', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None, 'shar

In [53]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

experiment = client.get_experiment_by_name("ChestScan_AI")

if experiment is None:
    raise ValueError("Expérience ChestScan_AI introuvable.")

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    max_results=100
)

metric_columns = [
    column for column in runs.columns
    if column.startswith("metrics.")
]

print("Métriques enregistrées :")
for column in metric_columns:
    print("-", column)

display(runs[["run_id", "tags.mlflow.runName"] + metric_columns])

ValueError: Expérience ChestScan_AI introuvable.

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# Évaluation du modèle sur le dataset de test
y_true = np.concatenate([
    labels.numpy().ravel()
    for _, labels in test_ds
])

y_prob = model.predict(test_ds, verbose=0).ravel()

# Si le modèle produit une probabilité pour la classe PNEUMONIA
y_pred = (y_prob >= 0.5).astype(int)

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc = roc_auc_score(y_true, y_prob)

print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-score  : {f1:.4f}")
print(f"AUC       : {auc:.4f}")

NameError: name 'accuracy' is not defined

In [ ]:
%python -m mlflow ui --backend-store-uri "sqlite:///artifacts/mlflow/mlflow.db" --host 127.0.0.1 --port 5000

UsageError: Line magic function `%python` not found (But cell magic `%%python` exists, did you mean that instead?).


## 4. Inspecter un run

In [ ]:
# None = le run le plus récent. Sinon, indique un run_id précis.
RUN_ID_TO_INSPECT = None

if not runs:
    raise RuntimeError("Aucun run trouvé dans cette expérience.")

selected_run = (
    runs[0] if RUN_ID_TO_INSPECT is None
    else client.get_run(RUN_ID_TO_INSPECT)
)

print("Run ID :", selected_run.info.run_id)
print("Nom :", selected_run.data.tags.get("mlflow.runName", "(sans nom)"))
print("Statut :", selected_run.info.status)

print("\nParamètres :")
display(pd.DataFrame(list(selected_run.data.params.items()),
                     columns=["paramètre", "valeur"]))

print("\nMétriques :")
display(pd.DataFrame(sorted(selected_run.data.metrics.items()),
                     columns=["métrique", "valeur"]))

print("\nTags :")
display(pd.DataFrame(list(selected_run.data.tags.items()),
                     columns=["tag", "valeur"]))


ValueError: The truth value of a DataFrame is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

## 5. Lister les artefacts enregistrés dans le run

In [ ]:
def list_artifacts_recursive(run_id, path=""):
    rows = []
    for item in client.list_artifacts(run_id, path):
        rows.append({
            "path": item.path,
            "is_dir": item.is_dir,
            "file_size_bytes": getattr(item, "file_size", None)
        })
        if item.is_dir:
            rows.extend(list_artifacts_recursive(run_id, item.path))
    return rows

artifacts_df = pd.DataFrame(list_artifacts_recursive(selected_run.info.run_id))
if artifacts_df.empty:
    print("Aucun artefact enregistré pour ce run.")
else:
    display(artifacts_df)


## 6. Lire les rapports CSV existants sans les modifier

In [ ]:
REPORT_DIR = PROJECT_DIR / "artifacts" / "reports"
COMPARISON_CSV = REPORT_DIR / "test_model_comparison.csv"
THRESHOLD_CSV = REPORT_DIR / "validation_threshold_results.csv"

print("Dossier des rapports :", REPORT_DIR, "| existe :", REPORT_DIR.exists())

if COMPARISON_CSV.exists():
    comparison_df = pd.read_csv(COMPARISON_CSV)
    print("\nComparaison des modèles sur le test :")
    display(comparison_df)
else:
    comparison_df = None
    print("Rapport absent :", COMPARISON_CSV)

if THRESHOLD_CSV.exists():
    threshold_df = pd.read_csv(THRESHOLD_CSV)
    print("\nRésultats des seuils sur la validation :")
    display(threshold_df)
else:
    threshold_df = None
    print("Rapport absent :", THRESHOLD_CSV)


## 7. Comparer les métriques présentes dans les runs

In [ ]:
metric_keys = [
    "accuracy", "precision", "recall", "f1", "specificity", "roc_auc",
    "test_accuracy", "test_precision", "test_recall", "test_f1",
    "test_specificity", "test_roc_auc"
]

comparison_rows = []
for run in runs:
    row = {
        "run_name": run.data.tags.get("mlflow.runName", ""),
        "run_id": run.info.run_id,
        "status": run.info.status,
    }
    for key in metric_keys:
        if key in run.data.metrics:
            row[key] = run.data.metrics[key]
    if any(key in row for key in metric_keys):
        comparison_rows.append(row)

mlflow_metrics_df = pd.DataFrame(comparison_rows)
if mlflow_metrics_df.empty:
    print("Aucune métrique correspondante trouvée. Consulte la table des runs de la section 3.")
else:
    display(mlflow_metrics_df)


## 8. Vérifier un run par son identifiant (facultatif)

In [ ]:
# Indique un run_id si tu veux vérifier un run particulier.
RUN_ID_TO_CHECK = ""

if RUN_ID_TO_CHECK.strip():
    run_check = client.get_run(RUN_ID_TO_CHECK.strip())
    print("Run :", run_check.info.run_id)
    print("Statut :", run_check.info.status)
    print("Métriques :", run_check.data.metrics)
    print("Paramètres :", run_check.data.params)
else:
    print("Aucun run_id fourni ; aucune action effectuée.")


## 9. Enregistrer un nouveau run avec des métriques déjà calculées (facultatif)

N'exécute cette cellule que si tu disposes de métriques réellement calculées. Elle ne lance pas d'entraînement.

**Ne choisis pas le seuil avec les résultats du test.** Choisis-le sur la validation, puis évalue le modèle une seule fois sur le test.


In [ ]:
NEW_RUN_NAME = "evaluation_manual_review"

# Remplace les exemples commentés par des valeurs réelles avant d'exécuter.
METRICS_TO_LOG = {
    # "test_accuracy": 0.80,
    # "test_precision": 0.78,
    # "test_recall": 0.99,
    # "test_f1": 0.87,
    # "test_roc_auc": 0.96,
}

if not METRICS_TO_LOG:
    print("Aucune métrique fournie : aucun run créé.")
else:
    with mlflow.start_run(run_name=NEW_RUN_NAME) as active_run:
        mlflow.set_tag("project", "ChestScan AI")
        mlflow.set_tag("source", "manual_existing_evaluation")
        mlflow.log_metrics({k: float(v) for k, v in METRICS_TO_LOG.items()})
        print("Nouveau run créé :", active_run.info.run_id)


## 10. Exporter un résumé des runs (facultatif)

In [ ]:
# Cette option écrit un nouveau fichier CSV ; elle ne remplace pas les rapports existants.
EXPORT_PATH = REPORT_DIR / "mlflow_runs_summary.csv"
EXPORT_RUNS_SUMMARY = False

if EXPORT_RUNS_SUMMARY:
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    runs_df.to_csv(EXPORT_PATH, index=False)
    print("Résumé exporté :", EXPORT_PATH)
else:
    print("Export désactivé. Passe EXPORT_RUNS_SUMMARY à True pour écrire :", EXPORT_PATH)


## 11. Résumé

- Les données sont lues depuis la base MLflow existante.
- Les runs, paramètres, métriques et artefacts sont consultables.
- Les rapports CSV existants sont lus sans être écrasés.
- Les sections d'enregistrement/export sont facultatives.
- **Aucun entraînement ni sauvegarde de modèle n'est effectué par ce notebook.**
